In [8]:
import json
import hashlib
from bs4 import BeautifulSoup
from collections import defaultdict

def find_duplicate_tiddlers(tiddlywiki_path, use_hash=False, hash_algo="md5"):
    """
    Identify tiddlers in a self-contained TiddlyWiki file that share identical text content.
    """
    with open(tiddlywiki_path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    tiddlers = []

    # Look for JSON script blocks
    for script_tag in soup.find_all("script", {"type": "application/json"}):
        content = (script_tag.string or "").strip()
        if not content:
            continue

        try:
            data = json.loads(content)
        except json.JSONDecodeError:
            continue

        # Case 1: Array of tiddlers
        if isinstance(data, list):
            for t in data:
                if isinstance(t, dict) and "title" in t:
                    tiddlers.append(t)

        # Case 2: Single tiddler object
        elif isinstance(data, dict) and "title" in data:
            tiddlers.append(data)

    print(f"✅ Found {len(tiddlers)} tiddlers")

    # --- Grouping and comparison
    text_map = defaultdict(list)
    for t in tiddlers:
        text_content = t.get("text", "").strip()
        title = t.get("title", "(untitled)")

        if use_hash:
            try:
                h = hashlib.new(hash_algo)
                h.update(text_content.encode("utf-8"))
                key = h.hexdigest()
            except ValueError:
                raise ValueError(f"Unsupported hash algorithm: {hash_algo}")
        else:
            key = text_content

        text_map[key].append(title)

    # --- Report duplicates
    duplicates = {k: v for k, v in text_map.items() if len(v) > 1}

    if not duplicates:
        print("⚠️ No duplicate text fields found")
    else:
        print(f"⚠️ Found {len(duplicates)} groups of duplicates:")
        for i, (k, titles) in enumerate(duplicates.items(), 1):
            print(f"\nGroup {i}: {len(titles)} tiddlers share identical content")
            for title in titles:
                print(f"  - {title}")

            if not use_hash:
                preview = k[:80].replace("\n", " ")
                print(f"  (Text preview: {preview!r}...)")
            else:
                print(f"  (Hash: {k})")

    return duplicates


In [14]:

path_to_wiki = "mywiki.html"  # Replace with your actual file path

# Normal comparison (no hashing)
find_duplicate_tiddlers(path_to_wiki)

# Hash-based comparison (MD5)
#find_duplicate_tiddlers(path_to_wiki, use_hash=True, hash_algo="md5")

# Hash-based comparison (SHA256)
#find_duplicate_tiddlers(path_to_wiki, use_hash=True, hash_algo="sha256")

✅ Found 34 tiddlers
⚠️ Found 4 groups of duplicates:

Group 1: 2 tiddlers share identical content
  - $:/isEncrypted
  - $:/status/RequireReloadDueToPluginChange
  (Text preview: 'no'...)

Group 2: 2 tiddlers share identical content
  - $:/state/notebook-sidebar
  - $:/state/showeditpreview
  (Text preview: 'yes'...)

Group 3: 3 tiddlers share identical content
  - compare-1
  - compare-1 1
  - compare-1 1 1
  (Text preview: 'Now if the time for all good men to come to the aide of their country'...)

Group 4: 3 tiddlers share identical content
  - compare-2
  - compare-2 1
  - compare-2 1 1
  (Text preview: 'True friends stab you in the front'...)


{'no': ['$:/isEncrypted', '$:/status/RequireReloadDueToPluginChange'],
 'yes': ['$:/state/notebook-sidebar', '$:/state/showeditpreview'],
 'Now if the time for all good men to come to the aide of their country': ['compare-1',
  'compare-1 1',
  'compare-1 1 1'],
 'True friends stab you in the front': ['compare-2',
  'compare-2 1',
  'compare-2 1 1']}